# Stratified train, validation, and test split

Split `df_with_color.parquet` into 60% training, 20% validation, and 20% test data. Both splits are stratified on `color`, so each set has approximately the same color distribution. The random seed makes the split reproducible. Keep the test set untouched until final evaluation.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 42
wine = pd.read_parquet('df_with_color.parquet')
assert wine['color'].notna().all(), 'Every row needs a color for stratification.'
print(f'Total reviews: {len(wine):,}')
print(wine['color'].value_counts().to_string())

Total reviews: 169,407
color
red          103819
white         54251
sparkling      5536
rose           4118
dessert        1453
other           230


First reserve 40% of the rows. Then split that reserved portion equally between validation and test, stratifying by `color` each time.

In [2]:
wine = wine[wine.color != 'other']
wine.color.unique()

<ArrowStringArray>
['red', 'white', 'rose', 'sparkling', 'dessert']
Length: 5, dtype: str

In [3]:
train, held_out = train_test_split(
    wine, test_size=0.40, random_state=SEED, stratify=wine['color']
)
validation, test = train_test_split(
    held_out, test_size=0.50, random_state=SEED, stratify=held_out['color']
)

assert len(train) + len(validation) + len(test) == len(wine)
assert train.index.is_unique and validation.index.is_unique and test.index.is_unique
assert train.index.intersection(validation.index).empty
assert train.index.intersection(test.index).empty
assert validation.index.intersection(test.index).empty

split_sizes = pd.DataFrame(
    {'rows': [len(train), len(validation), len(test)]},
    index=['train', 'validation', 'test'],
)
split_sizes['fraction'] = split_sizes['rows'] / len(wine)
display(split_sizes)

color_counts = pd.DataFrame({
    'full': wine['color'].value_counts(),
    'train': train['color'].value_counts(),
    'validation': validation['color'].value_counts(),
    'test': test['color'].value_counts(),
}).fillna(0).astype(int)
display(color_counts)
display(color_counts.div(color_counts.sum(axis=0), axis=1).round(4))

,rows,fraction
train,101506,0.599999
validation,33835,0.199998
test,33836,0.200004


,full,train,validation,test
color,,,,
red,103819,62291,20764,20764
white,54251,32550,10850,10851
sparkling,5536,3322,1107,1107
rose,4118,2471,823,824
dessert,1453,872,291,290


,full,train,validation,test
color,,,,
red,0.6137,0.6137,0.6137,0.6137
white,0.3207,0.3207,0.3207,0.3207
sparkling,0.0327,0.0327,0.0327,0.0327
rose,0.0243,0.0243,0.0243,0.0244
dessert,0.0086,0.0086,0.0086,0.0086


Save the three sets for use in later notebooks. The original row index is preserved so rows can be traced back to the source and checked for overlap.

In [4]:
train.to_parquet('train.parquet')
validation.to_parquet('validation.parquet')
test.to_parquet('test.parquet')
print('Saved train.parquet, validation.parquet, and test.parquet')

Saved train.parquet, validation.parquet, and test.parquet
